# Test CourtListener API access

Quick sanity check that `COURTLISTENER_API_TOKEN` is valid and `fetch_opinions()` can pull real data.

Requires `ipykernel` (`pip install ipykernel` if this kernel doesn't exist yet).

In [1]:
import os
import sys
from pathlib import Path

# Walk up from cwd to find the `ml/` dir (the one containing ingestion/courtlistener.py)
# so this works regardless of where Jupyter's working directory ends up.
candidate = Path.cwd().resolve()
ML_DIR = None
for parent in [candidate, *candidate.parents]:
    if (parent / "ingestion" / "courtlistener.py").exists():
        ML_DIR = parent
        break

if ML_DIR is None:
    raise RuntimeError(
        f"Could not locate ml/ingestion/courtlistener.py above {candidate}. "
        "Run this notebook from within the repo."
    )

sys.path.insert(0, str(ML_DIR))
print(f"ml/ dir: {ML_DIR}")

ml/ dir: C:\Users\aengu\legal-prediction\ml


In [2]:
# Load COURTLISTENER_API_TOKEN from the repo-root .env (no python-dotenv dependency needed)
ENV_PATH = ML_DIR.parent / ".env"

if ENV_PATH.exists():
    for line in ENV_PATH.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())
    print(f"Loaded env vars from {ENV_PATH}")
else:
    print(f"No .env found at {ENV_PATH} — set COURTLISTENER_API_TOKEN manually below if needed.")

token_present = bool(os.environ.get("COURTLISTENER_API_TOKEN"))
print(f"COURTLISTENER_API_TOKEN set: {token_present}")

Loaded env vars from C:\Users\aengu\legal-prediction\.env
COURTLISTENER_API_TOKEN set: True


In [3]:
from ingestion.courtlistener import fetch_opinions

# Small, cheap request just to confirm the token + request path work.
# "ca9" = 9th Circuit Court of Appeals; swap for any CourtListener court id.
results = fetch_opinions(court="ca9", page_size=5, max_pages=1)

print(f"Fetched {len(results)} opinion record(s)")

Fetched 20 opinion record(s)


In [4]:
import json

if results:
    print("Top-level keys on first record:")
    print(sorted(results[0].keys()))
    print()
    print(json.dumps(results[0], indent=2, default=str)[:2000])
else:
    print("No results returned — check the court id or your token's permissions.")

Top-level keys on first record:
['absolute_url', 'author', 'author_id', 'author_str', 'cluster', 'cluster_id', 'date_created', 'date_modified', 'download_url', 'extracted_by_ocr', 'html', 'html_anon_2020', 'html_columbia', 'html_lawbox', 'html_with_citations', 'id', 'joined_by', 'joined_by_str', 'local_path', 'main_version', 'opinions_cited', 'ordering_key', 'page_count', 'per_curiam', 'plain_text', 'resource_uri', 'sha1', 'type', 'xml_harvard', 'xml_scan']

{
  "resource_uri": "https://www.courtlistener.com/api/rest/v4/opinions/11402370/",
  "id": 11402370,
  "absolute_url": "/opinion/10934802/mendez-lemus-v-blanche/",
  "cluster_id": 10934802,
  "cluster": "https://www.courtlistener.com/api/rest/v4/clusters/10934802/",
  "author_id": null,
  "author": null,
  "joined_by": [],
  "date_created": "2026-07-24T10:00:48.877510-07:00",
  "date_modified": "2026-07-24T10:03:44.752198-07:00",
  "author_str": "",
  "per_curiam": false,
  "joined_by_str": "",
  "type": "010combined",
  "sha1": "